# Introducing LangChain

~ Open Source framework launched in October 2022

~ Provides a common framework for interfacing with many LLMs

~ LangChain v1 released in October 2025 with significant changes

### Pros & Cons

1. Simplifies the creation of applications using LLMs (eg: AI assistants, RAG, summarization) - fast time to market

2. Significant adoption in enterprise

3. As API for LLMs have matured, converged and simplified, the need for a unifying framework like LangChain has decreased - particularly with OpenAI endpoints

4. LangChain is a more heavyweight abstraction layer than LiteLLM and others, and some of its code can be 'legacy'

## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [14]:
#Imports

import os
import glob
import tiktoken
import numpy as np

from dotenv import load_dotenv

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_chroma import Chroma

from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [1]:
# Groq version

import os
import glob
import tiktoken
import numpy as np

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [15]:
#Gemini setup

MODEL = "gemini-2.5-flash-lite"

db_name = "vector_db"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key:
    print(
        f"Google API Key exists and begins "
        f"{google_api_key[:8]}"
    )
else:
    print("Google API Key not set")

Google API Key exists and begins AIzaSyCn


In [2]:
# Groq version

MODEL = "llama-3.3-70b-versatile"

db_name = "vector_db"

load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")

if groq_api_key:
    print(
        f"Groq API Key exists and begins "
        f"{groq_api_key[:8]}"
    )
else:
    print("Groq API Key not set")

Groq API Key exists and begins gsk_TdT2


In [16]:
#Create gemini LLM

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    temperature=0,
    google_api_key=google_api_key
)

response = llm.invoke(
    "Explain Artificial Intelligence in one sentence."
)

print(response.content)

Artificial Intelligence is the development of computer systems that can perform tasks typically requiring human intelligence, such as learning, problem-solving, and decision-making.


In [3]:
# Groq version

llm = ChatGroq(
    model=MODEL,
    temperature=0,
    groq_api_key=groq_api_key
)

response = llm.invoke(
    "What is Artificial Intelligence?"
)

print(response.content)

**Artificial Intelligence (AI)**: Artificial Intelligence refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

* Learning
* Problem-solving
* Reasoning
* Perception
* Understanding language

AI involves the use of algorithms, statistical models, and computer programs to enable machines to make decisions, classify objects, and predict outcomes without being explicitly programmed for each task.

**Key Characteristics of AI:**

1. **Machine Learning**: AI systems can learn from data and improve their performance over time.
2. **Autonomy**: AI systems can operate independently, making decisions without human intervention.
3. **Reasoning and Problem-Solving**: AI systems can analyze data, identify patterns, and make decisions based on that analysis.
4. **Natural Language Processing**: AI systems can understand, generate, and process human language.

**Types of AI:**

1. **Narrow or Weak AI**: Designed to perform a specific

In [17]:
# Read knowledge base

knowledge_base_path = "knowledge-base/**/*.md"

files = glob.glob(
    knowledge_base_path,
    recursive=True
)

print(
    f"Found {len(files)} files "
    f"in the knowledge base"
)

entire_knowledge_base = ""

for file_path in files:

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(
    f"Total characters in knowledge base: "
    f"{len(entire_knowledge_base):,}"
)

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [4]:
# Groq version

knowledge_base_path = "knowledge-base/**/*.md"

files = glob.glob(
    knowledge_base_path,
    recursive=True
)

print(
    f"Found {len(files)} files "
    f"in the knowledge base"
)

entire_knowledge_base = ""

for file_path in files:

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(
    f"Total characters in knowledge base: "
    f"{len(entire_knowledge_base):,}"
)

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [18]:
# Token count

encoding = tiktoken.get_encoding(
    "cl100k_base"
)

tokens = encoding.encode(
    entire_knowledge_base
)

token_count = len(tokens)

print(
    f"Approx token count: "
    f"{token_count:,}"
)

Approx token count: 63,721


In [5]:
# Groq version

encoding = tiktoken.get_encoding(
    "cl100k_base"
)

tokens = encoding.encode(
    entire_knowledge_base
)

token_count = len(tokens)

print(
    f"Approx token count: "
    f"{token_count:,}"
)

Approx token count: 63,721


In [19]:
# Load documents

folders = glob.glob(
    "knowledge-base/*"
)

documents = []

for folder in folders:

    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={
            "encoding": "utf-8"
        }
    )

    folder_docs = loader.load()

    for doc in folder_docs:

        doc.metadata["doc_type"] = doc_type

        documents.append(doc)

print(
    f"Loaded {len(documents)} documents"
)

Loaded 76 documents


In [6]:
# Groq version

folders = glob.glob(
    "knowledge-base/*"
)

documents = []

for folder in folders:

    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={
            "encoding": "utf-8"
        }
    )

    folder_docs = loader.load()

    for doc in folder_docs:

        doc.metadata["doc_type"] = doc_type

        documents.append(doc)

print(
    f"Loaded {len(documents)} documents"
)

Loaded 76 documents


In [20]:
documents[1]

Document(metadata={'source': 'knowledge-base\\company\\careers.md', 'doc_type': 'company'}, page_content="# Careers at Insurellm\n\n## Why Join Insurellm?\n\nAt Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.\n\nAfter reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic focus on high-value enterprise clients—from regional insurers to global reinsurance partners.\n\n### Our Culture\n\nWe live by our core values every day:\n- **Innovation First**: We encourage experimentation and creative problem-solving\

In [21]:
# Split documents

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(
    documents
)

print(
    f"Divided into "
    f"{len(chunks)} chunks"
)

print(
    f"First chunk:\n\n"
    f"{chunks[0]}"
)

Divided into 413 chunks
First chunk:

page_content='# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.' metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}


In [7]:
# Groq version

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(
    documents
)

print(
    f"Divided into "
    f"{len(chunks)} chunks"
)

print(chunks[0])

Divided into 413 chunks
page_content='# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.' metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}


In [22]:
chunks[100]

Document(metadata={'source': 'knowledge-base\\contracts\\Contract with GlobalRe Partners for Rellm.md', 'doc_type': 'contracts'}, page_content='13. **Climate Risk Analytics:** Forward-looking climate modeling:\n    - IPCC climate scenario analysis (RCP 2.6, 4.5, 8.5)\n    - Transition risk assessment\n    - Physical risk modeling for perils (hurricane, wildfire, flood, drought)\n    - Sea level rise impact analysis\n    - Temperature trend incorporation\n    - Climate-adjusted pricing recommendations\n    - Stranded asset identification\n    - Green reinsurance opportunities\n\n---\n\n## Support\n\nInsurellm commits to comprehensive Enterprise-level support for GlobalRe Partners:\n\n1. **Dedicated Success Team:**\n   - Executive sponsor (CEO-level) with quarterly strategic reviews\n   - Dedicated Senior Vice President of Customer Success with bi-weekly engagement\n   - Technical Account Manager for platform optimization\n   - Solutions Architect team (2 FTE) for strategic initiatives\n

# Encoder / Embedding Models

~ The Encoder Model turns text into a Vector Embedding. This is then stored in a Vector Database like Chroma.

1. word2vec [2013]

2. BERT [2018]

3. a. OpenAI: text embedding 3 small, text embedding 3 large

   b. Google: gemini-embedding-001
   
   c. Hugging Face Sentence Transformers: all-Mini-LM-L6-v2

### The Encoder turns the ttext into a Vector. The Vector store is a database that does quick lookup

Popular Vectorstores:

1. Open-source: Chroma, Qdrant, FAISS

2. Paid & scalable: Pinecone, Weaviate

3. Mainstream databases: Mongo, Postgres, Elastic

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [ ]:
# Pick an embedding model

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=google_api_key
)

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [8]:
# Groq version

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\KIIT\projects\llm_engineering\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KIIT\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Create Chroma DB

if os.path.exists(db_name):

    Chroma(
        persist_directory=db_name,
        embedding_function=embeddings
    ).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_name
)

print(
    f"Vectorstore created with "
    f"{vectorstore._collection.count()} "
    f"documents"
)

In [9]:
# Groq version

if os.path.exists(db_name):

    try:
        Chroma(
            persist_directory=db_name,
            embedding_function=embeddings
        ).delete_collection()
    except:
        pass

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_name
)

print(
    f"Vectorstore created with "
    f"{vectorstore._collection.count()} documents"
)

Vectorstore created with 413 documents


In [ ]:
# Vector dimensions check

collection = vectorstore._collection

count = collection.count()

sample_embedding = collection.get(
    limit=1,
    include=["embeddings"]
)["embeddings"][0]

dimensions = len(sample_embedding)

print(
    f"There are {count:,} vectors "
    f"with {dimensions:,} dimensions "
    f"in the vector store"
)

In [10]:
# Groq version

collection = vectorstore._collection

count = collection.count()

sample_embedding = collection.get(
    limit=1,
    include=["embeddings"]
)["embeddings"][0]

dimensions = len(sample_embedding)

print(
    f"There are {count:,} vectors "
    f"with {dimensions:,} dimensions"
)

There are 413 vectors with 384 dimensions


In [11]:
# Groq version - extract vectors

result = collection.get(
    include=[
        "embeddings",
        "documents",
        "metadatas"
    ]
)

vectors = np.array(result["embeddings"])

documents = result["documents"]

metadatas = result["metadatas"]

doc_types = [
    m["doc_type"]
    for m in metadatas
]

colors = [
    ['blue','green','red','orange'][
        ['products','employees','contracts','company']
        .index(t)
    ]
    for t in doc_types
]

### Part C: Visualize!

In [ ]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

In [12]:
# Groq version

result = collection.get(
    include=[
        "embeddings",
        "documents",
        "metadatas"
    ]
)

vectors = np.array(result["embeddings"])

documents = result["documents"]

metadatas = result["metadatas"]

doc_types = [
    m["doc_type"]
    for m in metadatas
]

colors = [
    ['blue','green','red','orange'][
        ['products','employees','contracts','company']
        .index(t)
    ]
    for t in doc_types
]

In [14]:
# Groq version

tsne = TSNE(
    n_components=2,
    random_state=42
)

reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:,0],
            y=reduced_vectors[:,1],
            mode='markers',
            marker=dict(
                size=5,
                color=colors,
                opacity=0.8
            ),
            text=[
                f"Type:{t}<br>{d[:100]}"
                for t,d in zip(
                    doc_types,
                    documents
                )
            ],
            hoverinfo='text'
        )
    ]
)

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    xaxis_title='TSNE Dimension 1',
    yaxis_title='TSNE Dimension 2',
    width=900,
    height=700
)

fig.show()

In [13]:
# Groq version

tsne = TSNE(
    n_components=3,
    random_state=42
)

reduced_vectors = tsne.fit_transform(
    vectors
)

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:,0],
            y=reduced_vectors[:,1],
            z=reduced_vectors[:,2],
            mode='markers',
            marker=dict(
                size=5,
                color=colors,
                opacity=0.8
            ),
            text=[
                f"Type:{t}<br>{d[:100]}"
                for t,d in zip(
                    doc_types,
                    documents
                )
            ],
            hoverinfo='text'
        )
    ]
)

fig.show()